In [1]:
# Q2 - Age and life stage

# Question: which age groups say yes to a term deposit most often?

# Each age band is compared with the overall yes rate (the Q1 baseline). Every rate is shown with its record count; groups under 100 records are flagged and not interpreted.

In [2]:
import pandas as pd

CLEAN_PATH = "../../data/bank_marketing_clean.parquet"
clean_df = pd.read_parquet(CLEAN_PATH)

clean_df.shape

(41176, 22)

In [3]:
# Q2, step 1: yes rate by age band
subscribed = clean_df["subscribed_term_deposit"] == "yes"
baseline_rate = subscribed.mean() * 100

age_edges = [0, 25, 35, 45, 55, 65, 120]
age_labels = ["under 25", "25-34", "35-44", "45-54", "55-64", "65 and over"]
age_band = pd.cut(clean_df["age"], bins=age_edges, labels=age_labels, right=False)

summary = (
    pd.DataFrame({"age_band": age_band, "subscribed": subscribed})
    .groupby("age_band", observed=True)["subscribed"]
    .agg(customers="count", subscribers="sum")
)
summary["yes_rate_pct"] = (summary["subscribers"] / summary["customers"] * 100).round(2)
summary["vs_baseline_points"] = (summary["yes_rate_pct"] - baseline_rate).round(2)
summary["share_of_customers_pct"] = (summary["customers"] / len(clean_df) * 100).round(2)
summary["share_of_subscribers_pct"] = (summary["subscribers"] / subscribed.sum() * 100).round(2)
summary["small_group"] = summary["customers"] < 100

print(f"Baseline yes rate: {baseline_rate:.2f}%")
print(f"Customers in bands: {summary['customers'].sum()} (should be {len(clean_df)})")
print(f"Subscribers in bands: {summary['subscribers'].sum()} (should be {subscribed.sum()})")
print()
summary

Baseline yes rate: 11.27%
Customers in bands: 41176 (should be 41176)
Subscribers in bands: 4639 (should be 4639)



,customers,subscribers,yes_rate_pct,vs_baseline_points,share_of_customers_pct,share_of_subscribers_pct,small_group
age_band,,,,,,,
under 25,1067,256,23.99,12.72,2.59,5.52,False
25-34,13684,1666,12.17,0.90,33.23,35.91,False
35-44,13495,1168,8.66,-2.61,32.77,25.18,False
45-54,8702,752,8.64,-2.63,21.13,16.21,False
55-64,3566,484,13.57,2.30,8.66,10.43,False
65 and over,662,313,47.28,36.01,1.61,6.75,False


In [4]:
# Q2, step 2: five-year age steps, to see where the high rates at the ends come from
fine_edges = [15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 120]
fine_labels = ["15-19", "20-24", "25-29", "30-34", "35-39", "40-44", "45-49",
               "50-54", "55-59", "60-64", "65-69", "70-74", "75-79", "80 and over"]
fine_band = pd.cut(clean_df["age"], bins=fine_edges, labels=fine_labels, right=False)

fine_summary = (
    pd.DataFrame({"age_band": fine_band, "subscribed": subscribed})
    .groupby("age_band", observed=True)["subscribed"]
    .agg(customers="count", subscribers="sum")
)
fine_summary["yes_rate_pct"] = (fine_summary["subscribers"] / fine_summary["customers"] * 100).round(2)
fine_summary["vs_baseline_points"] = (fine_summary["yes_rate_pct"] - baseline_rate).round(2)
fine_summary["small_group"] = fine_summary["customers"] < 100

print(f"Customers in steps: {fine_summary['customers'].sum()} (should be {len(clean_df)})")
print(f"Subscribers in steps: {fine_summary['subscribers'].sum()} (should be {subscribed.sum()})")
for label, parts in [("under 25", ["15-19", "20-24"]),
                     ("65 and over", ["65-69", "70-74", "75-79", "80 and over"])]:
    part_customers = fine_summary.loc[parts, "customers"].sum()
    part_subscribers = fine_summary.loc[parts, "subscribers"].sum()
    print(f"{label}: {part_customers} customers, {part_subscribers} subscribers "
          f"(Cell 3: {summary.loc[label, 'customers']}, {summary.loc[label, 'subscribers']})")
print()
fine_summary

Customers in steps: 41176 (should be 41176)
Subscribers in steps: 4639 (should be 4639)
under 25: 1067 customers, 256 subscribers (Cell 3: 1067, 256)
65 and over: 662 customers, 313 subscribers (Cell 3: 662, 313)



,customers,subscribers,yes_rate_pct,vs_baseline_points,small_group
age_band,,,,,
15-19,75,34,45.33,34.06,True
20-24,992,222,22.38,11.11,False
25-29,4600,666,14.48,3.21,False
30-34,9084,1000,11.01,-0.26,False
35-39,7849,715,9.11,-2.16,False
40-44,5646,453,8.02,-3.25,False
45-49,4877,380,7.79,-3.48,False
50-54,3825,372,9.73,-1.54,False
55-59,3036,325,10.70,-0.57,False
